In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

In [2]:
training_data = pd.read_csv('/content/training_data_final (1).csv')
print(training_data.shape)
print(training_data.head())

(4109, 8)
   State    District   Latitude  Longitude  slope_deg  rainfall_mm  \
0  Assam  Hailakandi  24.515944  92.540889       0.26    12.183607   
1  Assam  Hailakandi  24.511306  92.693028       1.65    14.154098   
2  Assam  Hailakandi  24.703640  92.646810       1.03    11.861475   
3  Assam  Hailakandi  24.706560  92.650080       0.93    11.861475   
4  Assam   Karimganj  24.710833  92.481167       0.67    14.931967   

   soil_moisture_mm  label  
0          0.411990      1  
1          0.411175      1  
2          0.492546      1  
3          0.492546      1  
4          0.484775      1  


In [3]:
feature_columns = ['Latitude', 'Longitude', 'slope_deg', 'rainfall_mm', 'soil_moisture_mm']
X = training_data[feature_columns]
y = training_data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train size:", X_train.shape, " Test size:", X_test.shape)

Train size: (3287, 5)  Test size: (822, 5)


In [4]:
# Train binary model using RandomForest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [5]:
# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9537712895377128
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       410
           1       0.95      0.95      0.95       412

    accuracy                           0.95       822
   macro avg       0.95      0.95      0.95       822
weighted avg       0.95      0.95      0.95       822

[[391  19]
 [ 19 393]]


In [6]:
# Assign risk levels using quantile-based thresholds
probs = model.predict_proba(X)[:, 1]
training_data['risk_probability'] = probs

# save actual threshold VALUES (not just rank-based labels) for reuse on new predictions
risk_thresholds = training_data['risk_probability'].quantile([0.25, 0.5, 0.75]).to_dict()
print("Risk thresholds:", risk_thresholds)

training_data['risk_level'] = pd.qcut(
    training_data['risk_probability'].rank(method='first'),
    q=4,
    labels=['Low', 'Medium', 'High', 'Critical']
)

print(training_data['risk_level'].value_counts())
print(training_data[['District', 'risk_probability', 'risk_level']].head(10))


Risk thresholds: {0.25: 0.01, 0.5: 0.53, 0.75: 1.0}
risk_level
Low         1028
Medium      1027
High        1027
Critical    1027
Name: count, dtype: int64
     District  risk_probability risk_level
0  Hailakandi              0.76       High
1  Hailakandi              0.45     Medium
2  Hailakandi              1.00       High
3  Hailakandi              1.00       High
4   Karimganj              0.90       High
5      Cachar              0.88       High
6      Cachar              0.75       High
7       Anjaw              0.83       High
8       Anjaw              0.95       High
9       Anjaw              1.00       High


In [7]:
# Save final labeled dataset (with risk levels) for reference/dashboard
training_data.to_csv('training_data_with_risk_levels.csv', index=False)
print("Saved:", training_data.shape)

# save thresholds too, so backend/Person 2 can replicate risk-level logic
import json
with open('risk_thresholds.json', 'w') as f:
    json.dump(risk_thresholds, f)
print("Thresholds saved:", risk_thresholds)

Saved: (4109, 10)
Thresholds saved: {0.25: 0.01, 0.5: 0.53, 0.75: 1.0}


In [8]:
# Save model
import pickle
with open('landslide_risk_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved!")

Model saved!


In [9]:
# Full pipeline self-test — from raw features to human-readable risk level
def predict_full_risk(lat, lon, slope, rainfall, soil_moisture, model, thresholds):
    input_df = pd.DataFrame([{
        'Latitude': lat,
        'Longitude': lon,
        'slope_deg': slope,
        'rainfall_mm': rainfall,
        'soil_moisture_mm': soil_moisture
    }])
    prob = model.predict_proba(input_df)[:, 1][0]

    if prob <= thresholds[0.25]:
        risk = 'Low'
    elif prob <= thresholds[0.5]:
        risk = 'Medium'
    elif prob < thresholds[0.75]:
        risk = 'High'
    else:
        risk = 'Critical'

    return {'probability': round(prob, 3), 'risk_level': risk}

# reload model to confirm it works fresh, like Person 2 will do
with open('landslide_risk_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

sample_row = training_data.iloc[636]
result = predict_full_risk(
    sample_row['Latitude'], sample_row['Longitude'],
    sample_row['slope_deg'], sample_row['rainfall_mm'], sample_row['soil_moisture_mm'],
    loaded_model, risk_thresholds
)

print("State:", sample_row['State'], " District:", sample_row['District'])
print(result)

State: Assam  District: Dima Hasao
{'probability': np.float64(1.0), 'risk_level': 'Critical'}
